<a href="https://colab.research.google.com/github/lautarodibartolo-ae/e8102-exploracion-de-datos/blob/main/bloque-1-probabilidad-y-variables-aleatorias/u1_ejemplos_probabilidad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>


# Unidad 1 — Probabilidad, en ejemplos

**Asignatura E8102 · Exploración de datos**

Este notebook acompaña al apunte de la unidad 1. No agrega teoría: toma los tres ejemplos del
apunte, el mazo de 40 cartas, el bolillero de 5 blancas y 3 negras y el relevamiento de 37 hogares,
y los deja tocar. Cada bloque dice a qué sección del apunte corresponde y qué conviene mirar.

Se lee el apunte primero y se juega con esto después.


## Cómo se usa

Antes de tocar nada: `Archivo` → `Guardar una copia en Drive`. Si no, los cambios se pierden.

Este notebook no pide escribir código. Cada bloque tiene un **formulario** con deslizadores y
listas desplegables. Se mueve un valor y el gráfico se vuelve a dibujar solo.

Tres pasos:

1. Ejecutar todo una vez: `Entorno de ejecución` → `Ejecutar todo`. Tarda unos segundos y deja
   listas las herramientas y los datos.
2. Bajar hasta un bloque, leer el resumen y mover los parámetros del formulario. La celda se
   ejecuta sola con cada cambio.
3. Si un formulario no reacciona, hacer clic adentro y apretar `Shift + Enter`, o volver al paso 1.

El código de cada formulario está oculto a propósito. Si te da curiosidad, `Mostrar código` lo
despliega, pero nada de este notebook requiere leerlo.


## Preparación

Dos celdas. La primera carga las herramientas y la paleta de colores. La segunda carga el
relevamiento de hogares del T8001, que los dos apuntes usan como ejemplo con datos reales.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.transforms as mtransforms
from matplotlib.patches import Circle, Polygon, Rectangle
from cycler import cycler
from fractions import Fraction
from io import StringIO
import scipy
from scipy import stats

# La paleta de las ilustraciones del apunte, para que los gráficos se vean como el mismo material.
CREMA, TEAL, TEAL_PALIDO, MOSTAZA, CORAL, TINTA = (
    "#FEF8EF", "#61A9A1", "#D0E5DE", "#EEA92F", "#F07C52", "#32435D")

plt.rcParams.update({
    "figure.facecolor": CREMA, "axes.facecolor": CREMA, "savefig.facecolor": CREMA,
    "axes.edgecolor": TINTA, "axes.labelcolor": TINTA, "text.color": TINTA,
    "xtick.color": TINTA, "ytick.color": TINTA,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.prop_cycle": cycler(color=[TEAL, CORAL, MOSTAZA, TINTA]),
    "axes.grid": True, "grid.color": TEAL_PALIDO, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "figure.figsize": (9, 4.5), "figure.dpi": 100,
    "axes.titlesize": 13, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "legend.frameon": False, "font.size": 11,
})


def num(x, decimales=4):
    """Número con coma decimal, como en el apunte."""
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return "—"
    if abs(x) < 0.5 * 10 ** -decimales:
        x = 0.0   # evita el "-0,0000"
    return f"{x:,.{decimales}f}".replace(",", "X").replace(".", ",").replace("X", ".")


def frac(k, n):
    """'k/n = 0,xxxx' con la fracción sin simplificar, que es como la escribe el apunte."""
    return f"{k}/{n} = {num(k / n)}"


print("numpy", np.__version__, "| pandas", pd.__version__, "| scipy", scipy.__version__, "| todo listo")


El relevamiento va adentro del notebook, como texto, así que no hace falta conexión. Son **45
visitas a 37 hogares**: ocho hogares recibieron una segunda visita. El apunte de la unidad 2 fija la
regla, la primera visita de cada hogar, y acá se usa la misma en las dos unidades. La copia
publicada está en
[`datos/relevamiento_plano.csv`](https://github.com/lautarodibartolo-ae/e8102-exploracion-de-datos/blob/main/bloque-1-probabilidad-y-variables-aleatorias/datos/relevamiento_plano.csv).


In [ ]:
CSV = """id_hogar,localidad,provincia,fecha_visita,encuestador_legajo,encuestador_nombre,encuestador_telefono,personas,ingreso,cobertura_salud,satisfaccion,servicios
1,Ramallo,Buenos Aires,2025-03-14,E04,"Sosa, Diego",3407-419854,3,185000,si,buena,agua;luz;gas
1,Ramallo,Buenos Aires,2025-06-12,E04,"Sosa, Diego",3407-419854,4,201000,si,buena,agua;luz;gas
2,Córdoba,Córdoba,2025-03-15,E02,"Ledesma, Julio",351-4778120,5,240500,no,regular,agua;luz
3,Concepción,Tucumán,2025-03-15,E03,"Bianchi, Ana",381-4551907,2,,,,luz
4,Ramallo,Buenos Aires,2025-03-16,E01,"Rivas, Marta",3407-412233,4,198000,si,buena,agua;luz;gas
5,Ramallo,Buenos Aires,2025-03-16,E04,"Sosa, Diego",3407-419854,1,120000,,,luz
7,Córdoba,Córdoba,2025-03-17,E04,"Sosa, Diego",3407-419854,6,310000,no,mala,agua;luz;gas;cloacas
7,Córdoba,Córdoba,2025-06-13,E04,"Sosa, Diego",3407-419854,6,325000,no,regular,agua;luz;gas;cloacas
8,Concepción,Tucumán,2025-03-17,E03,"Bianchi, Ana",381-4551907,3,175000,si,regular,agua;luz
9,Ramallo,Buenos Aires,2025-03-18,E04,"Sosa, Diego",3407-419854,2,,no,mala,luz;gas
10,Córdoba,Córdoba,2025-03-19,E02,"Ledesma, Julio",351-4778120,4,205000,,,agua;luz;gas
11,Concepción,Tucumán,2025-03-20,E03,"Bianchi, Ana",381-4551907,7,260000,si,buena,agua;luz;gas;cloacas
11,Concepción,Tucumán,2025-06-13,E03,"Bianchi, Ana",381-4551907,7,268000,si,buena,agua;luz;gas;cloacas
13,Ramallo,Buenos Aires,2025-03-20,E04,"Sosa, Diego",3407-419854,3,190000,,,agua;luz
14,Córdoba,Córdoba,2025-03-21,E02,"Ledesma, Julio",351-4778120,2,168000,no,regular,agua
15,Ramallo,Buenos Aires,2025-03-21,E04,"Sosa, Diego",3407-419854,3,172000,si,regular,agua;luz;gas
16,Concepción,Tucumán,2025-03-24,E03,"Bianchi, Ana",381-4551907,4,195000,no,buena,agua;luz
17,Córdoba,Córdoba,2025-03-24,E04,"Sosa, Diego",3407-419854,2,210000,si,regular,agua;luz;gas
18,Córdoba,Córdoba,2025-03-25,E02,"Ledesma, Julio",351-4778120,5,155000,,,luz
19,Ramallo,Buenos Aires,2025-03-25,E04,"Sosa, Diego",3407-419854,3,188000,no,mala,agua;luz
20,Concepción,Tucumán,2025-03-26,E03,"Bianchi, Ana",381-4551907,6,225000,si,buena,agua;luz;gas;cloacas
20,Concepción,Tucumán,2025-06-16,E03,"Bianchi, Ana",381-4551907,5,231000,si,regular,agua;luz;gas;cloacas
21,Ramallo,Buenos Aires,2025-03-26,E04,"Sosa, Diego",3407-419854,2,163000,,,agua;luz
22,Córdoba,Córdoba,2025-03-27,E02,"Ledesma, Julio",351-4778120,4,201000,si,regular,agua;luz;gas
23,Ramallo,Buenos Aires,2025-03-27,E04,"Sosa, Diego",3407-419854,3,179000,no,buena,agua;luz
24,Concepción,Tucumán,2025-03-28,E03,"Bianchi, Ana",381-4551907,5,232000,si,regular,agua;luz;gas
25,Córdoba,Córdoba,2025-03-28,E04,"Sosa, Diego",3407-419854,1,148000,,,luz
26,Ramallo,Buenos Aires,2025-03-31,E01,"Rivas, Marta",3407-412233,4,216000,no,regular,agua;luz;gas
26,Ramallo,Buenos Aires,2025-06-17,E01,"Rivas, Marta",3407-412233,4,222000,no,buena,agua;luz;gas
27,Concepción,Tucumán,2025-03-31,E03,"Bianchi, Ana",381-4551907,2,193000,si,mala,agua;luz
30,Ramallo,Buenos Aires,2025-04-01,E01,"Rivas, Marta",3407-412233,7,244000,si,buena,agua;luz;gas;cloacas
30,Ramallo,Buenos Aires,2025-06-18,E01,"Rivas, Marta",3407-412233,8,259000,si,buena,agua;luz;gas;cloacas
31,Concepción,Tucumán,2025-04-02,E03,"Bianchi, Ana",381-4551907,4,181000,,,agua;luz
32,Córdoba,Córdoba,2025-04-02,E02,"Ledesma, Julio",351-4778120,3,207000,no,regular,agua;luz;gas
33,Ramallo,Buenos Aires,2025-04-02,E04,"Sosa, Diego",3407-419854,5,169000,si,regular,agua;luz
34,Concepción,Tucumán,2025-04-03,E03,"Bianchi, Ana",381-4551907,2,236000,no,buena,agua;luz;gas
35,Córdoba,Córdoba,2025-04-03,E04,"Sosa, Diego",3407-419854,4,152000,,,agua
36,Ramallo,Buenos Aires,2025-04-04,E01,"Rivas, Marta",3407-412233,6,199000,si,mala,agua;luz;gas
36,Ramallo,Buenos Aires,2025-06-19,E01,"Rivas, Marta",3407-412233,6,204000,si,regular,agua;luz;gas
37,Concepción,Tucumán,2025-04-04,E03,"Bianchi, Ana",381-4551907,3,223000,no,regular,agua;luz
38,Córdoba,Córdoba,2025-04-07,E02,"Ledesma, Julio",351-4778120,2,0,si,mala,luz
39,Ramallo,Buenos Aires,2025-04-07,E04,"Sosa, Diego",3407-419854,12,620000,no,buena,agua;luz;gas;cloacas
39,Ramallo,Buenos Aires,2025-06-20,E04,"Sosa, Diego",3407-419854,12,641000,no,regular,agua;luz;gas;cloacas
40,Concepción,Tucumán,2025-04-08,E03,"Bianchi, Ana",381-4551907,4,186000,,,agua;luz
42,Ramallo,Buenos Aires,2025-04-09,E01,"Rivas, Marta",3407-412233,2,176000,no,mala,agua;luz"""

visitas = pd.read_csv(StringIO(CSV), parse_dates=["fecha_visita"])
primera = (visitas.sort_values("fecha_visita").drop_duplicates("id_hogar", keep="first")
           .sort_values("id_hogar").reset_index(drop=True))
segunda = (visitas.sort_values("fecha_visita").drop_duplicates("id_hogar", keep="last")
           .sort_values("id_hogar").reset_index(drop=True))
hogares = primera   # la regla del apunte: un hogar, su primera visita

print(f"{len(visitas)} visitas, {len(hogares)} hogares, {hogares['localidad'].nunique()} localidades")
print("personas por hogar, primera visita:",
      hogares["personas"].value_counts().sort_index().to_dict())


---
# 1. Espacio muestral, evento y complemento

*Apunte, secciones 2.2 a 2.5.*

El mazo español tiene 40 cartas, y ese es el espacio muestral: cada carta es un resultado posible.
Un **evento** es un grupo de esos resultados, definido por una condición. Su probabilidad, con el
enfoque clásico, es la cantidad de cartas que cumplen la condición sobre 40. El complemento es lo
que queda sin pintar, y las dos probabilidades suman 1.

Tres combinaciones para probar: `espadas` y `figura` es el `3/40` que el apunte repite; `espadas` y
`cualquiera` es el `10/40` de la sección 3.1; `cualquiera` y `cualquiera` pinta todo el mazo, y es
el evento seguro de la sección 2.4, con complemento `0/40`.


In [ ]:
#@title El mazo como espacio muestral { run: "auto", display-mode: "form" }

palo = "espadas"  #@param ["cualquiera", "oros", "copas", "espadas", "bastos"]
tipo_de_carta = "figura"  #@param ["cualquiera", "figura", "número"]

PALOS = ["oros", "copas", "espadas", "bastos"]
VALORES = [1, 2, 3, 4, 5, 6, 7, 10, 11, 12]
NOMBRE = {10: "S", 11: "C", 12: "R"}
mazo = [(p, v) for p in PALOS for v in VALORES]


def favorable(carta):
    p, v = carta
    ok_palo = palo == "cualquiera" or p == palo
    ok_tipo = tipo_de_carta == "cualquiera" or (tipo_de_carta == "figura") == (v >= 10)
    return ok_palo and ok_tipo


k, n = sum(favorable(c) for c in mazo), len(mazo)

fig, ax = plt.subplots(figsize=(10, 4.2))
for i, p in enumerate(PALOS):
    for j, v in enumerate(VALORES):
        es = favorable((p, v))
        ax.add_patch(Rectangle((j, 3 - i), 0.92, 0.92, facecolor=CORAL if es else TEAL_PALIDO,
                               edgecolor=TINTA, linewidth=1.2))
        ax.text(j + 0.46, 3 - i + 0.46, NOMBRE.get(v, str(v)), ha="center", va="center",
                fontsize=11, color="white" if es else TINTA, weight="bold" if es else "normal")
ax.set_xlim(-0.1, 10.1); ax.set_ylim(-0.1, 4.1)
ax.set_yticks([3.46, 2.46, 1.46, 0.46]); ax.set_yticklabels(PALOS)
ax.set_xticks([]); ax.grid(False)
for lado in ax.spines.values():
    lado.set_visible(False)
ax.set_title(f"El evento ocupa {k} de las {n} cartas")
plt.show()

print(f"Casos favorables: {k}   Casos posibles: {n}")
print(f"P(evento)      = {frac(k, n)} = {num(100 * k / n, 1)}%")
print(f"P(complemento) = {frac(n - k, n)}")
print(f"Suma:            {num(k / n + (n - k) / n)}")


---
# 2. El enfoque empírico y la estabilidad de las frecuencias

*Apunte, secciones 3.2 y 3.5.*

El enfoque clásico da un número sin tirar nada: casos favorables sobre posibles. El empírico repite
el experimento y cuenta. El **principio de estabilidad** dice que la frecuencia relativa, con muchas
repeticiones, se aplaca alrededor de la probabilidad.

Tres paneles, y los tres muestran una advertencia del apunte. A la izquierda, varias series con la
misma probabilidad: con pocas tiradas están lejos y cada una por su lado; con miles se juntan sobre
la línea. En el medio, la **diferencia absoluta** entre los éxitos que salieron y los que "debían"
salir: no se achica, crece. Lo que se aplaca es la proporción, no el desbalance. A la derecha, la
**falacia del jugador**: después de una racha de éxitos, la frecuencia del siguiente no baja. Las
dos barras salen de la misma simulación y las dos oscilan alrededor de la probabilidad.

Con la semilla se cambia el azar. Con `0,50` y `20000` tiradas se reproduce el ejemplo de la moneda.


In [ ]:
#@title Frecuencia relativa, desbalance absoluto y falacia del jugador { run: "auto", display-mode: "form" }

probabilidad = 0.25  #@param {type:"slider", min:0.05, max:0.95, step:0.05}
tiradas = 1000  #@param {type:"slider", min:100, max:20000, step:100}
series = 3  #@param {type:"slider", min:1, max:6, step:1}
racha = 3  #@param {type:"slider", min:1, max:6, step:1}
semilla = 1  #@param {type:"slider", min:0, max:100, step:1}

rng = np.random.default_rng(semilla)
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4.4), gridspec_kw={"width_ratios": [2, 1.4, 1]})

n = np.arange(1, tiradas + 1)
for _ in range(series):
    exitos = rng.random(tiradas) < probabilidad
    acumulados = np.cumsum(exitos)
    ax1.plot(n, acumulados / n, linewidth=1.3, alpha=0.9)
    ax2.plot(n, acumulados - n * probabilidad, linewidth=1.3, alpha=0.9)
ax1.axhline(probabilidad, color=TINTA, linestyle="--", linewidth=1.2)
ax1.set_xscale("log"); ax1.set_xlim(10, tiradas); ax1.set_ylim(0, 1)
ax1.set_xlabel("cantidad de repeticiones (escala logarítmica)")
ax1.set_ylabel("frecuencia relativa del éxito")
ax1.set_title(f"La proporción se aplaca hacia {num(probabilidad, 2)}")
ax2.axhline(0, color=TINTA, linestyle="--", linewidth=1.2)
ax2.set_xlabel("cantidad de repeticiones"); ax2.set_ylabel("éxitos de más (+) o de menos (−)")
ax2.set_title("El desbalance absoluto no se achica")

largo = 200_000
exitos = rng.random(largo) < probabilidad
ventana = np.lib.stride_tricks.sliding_window_view(exitos[:-1], racha).all(axis=1)
siguientes = exitos[racha:][ventana]
sin_condicion = exitos.mean()
frecuencia = siguientes.mean() if len(siguientes) >= 100 else np.nan
ax3.bar(["sin condición", f"tras {racha} éxito(s)\nseguidos"], [sin_condicion, np.nan_to_num(frecuencia)],
        color=[TEAL, CORAL], width=0.6)
ax3.axhline(probabilidad, color=TINTA, linestyle="--", linewidth=1.2)
ax3.set_ylim(0, 1); ax3.set_title("El azar no tiene memoria")
for i, v in enumerate([sin_condicion, frecuencia]):
    ax3.text(i, (0 if np.isnan(v) else v) + 0.02, num(v, 3) if not np.isnan(v) else "muy pocas\nrachas", ha="center")
plt.tight_layout(); plt.show()

miles = lambda n: f"{n:,}".replace(",", ".")
print(f"Simulación larga: {miles(largo)} tiradas, {miles(int(exitos.sum()))} éxitos, frecuencia {num(sin_condicion, 3)}.")
if np.isnan(frecuencia):
    print(f"Con probabilidad {num(probabilidad, 2)} casi no aparecen {racha} éxitos seguidos ({len(siguientes)} rachas).")
    print("Subí la probabilidad o bajá la racha para que la barra de la derecha tenga sentido.")
else:
    print(f"Hubo {miles(len(siguientes))} rachas de {racha} éxito(s) seguidos. Frecuencia del éxito justo después: {num(frecuencia, 3)}.")
    print(f"Las dos frecuencias se mueven alrededor de {num(probabilidad, 2)}. La diferencia entre ellas es ruido de la"
          " simulación: cambiá la semilla y mirá cómo cambia de signo.")


---
# 3. Independientes o dependientes: con y sin reposición

*Apunte, secciones 4.2 y 4.3.*

Del bolillero se sacan dos bolillas. Si la primera **vuelve** antes de sacar la segunda, el
bolillero está igual que al principio, y saber qué salió primero no cambia nada: los eventos son
**independientes**. Si **no vuelve**, el bolillero cambió, y la probabilidad de la segunda depende
de qué salió.

El gráfico muestra la probabilidad de que la segunda sea blanca en dos situaciones: sabiendo que la
primera fue blanca, y sabiendo que fue negra. Con reposición las dos barras son iguales: da lo mismo
qué salió. Sin reposición son distintas, y esa diferencia es la dependencia. Con 5 blancas y 3
negras son el `4/7` y el `5/7` del apunte. Abajo, los cuatro caminos sobre 56, que es el denominador
de toda la unidad.


In [ ]:
#@title El bolillero, con y sin reposición { run: "auto", display-mode: "form" }

blancas = 5  #@param {type:"slider", min:1, max:12, step:1}
negras = 3  #@param {type:"slider", min:1, max:12, step:1}

total = blancas + negras
con = {"blanca": (blancas, total), "negra": (blancas, total)}
sin = {"blanca": (blancas - 1, total - 1), "negra": (blancas, total - 1)}

fig, ax = plt.subplots(figsize=(9, 4.6))
x = np.arange(2)
valores = [con["blanca"], con["negra"], sin["blanca"], sin["negra"]]
ax.bar(x - 0.19, [con["blanca"][0] / total, sin["blanca"][0] / (total - 1)], 0.38, color=TEAL,
       label="P(2ª blanca | 1ª blanca)")
ax.bar(x + 0.19, [con["negra"][0] / total, sin["negra"][0] / (total - 1)], 0.38, color=CORAL,
       label="P(2ª blanca | 1ª negra)")
for xx, (k, d) in zip([x[0] - 0.19, x[1] - 0.19, x[0] + 0.19, x[1] + 0.19],
                      [con["blanca"], sin["blanca"], con["negra"], sin["negra"]]):
    ax.text(xx, k / d + 0.02, f"{k}/{d}", ha="center")
ax.set_xticks(x); ax.set_xticklabels(["con reposición", "sin reposición"])
ax.set_ylim(0, 1.25); ax.legend(loc="upper center", ncol=2)
ax.set_title(f"{blancas} blancas y {negras} negras: la reposición decide la dependencia")
plt.show()

print("Con reposición:  P(2ª blanca | 1ª blanca) = " + frac(*con["blanca"]) +
      "   P(2ª blanca | 1ª negra) = " + frac(*con["negra"]) + "   → iguales: independientes")
print("Sin reposición:  P(2ª blanca | 1ª blanca) = " + frac(*sin["blanca"]) +
      "   P(2ª blanca | 1ª negra) = " + frac(*sin["negra"]) +
      ("   → distintas: dependientes" if sin["blanca"][0] != sin["negra"][0] else "   → iguales"))

den = total * (total - 1)
caminos = {
    "blanca, blanca": blancas * (blancas - 1),
    "blanca, negra": blancas * negras,
    "negra, blanca": negras * blancas,
    "negra, negra": negras * (negras - 1),
}
print(f"\nLos cuatro caminos sin reposición, sobre {total} × {total - 1} = {den}:")
for nombre, numerador in caminos.items():
    print(f"  {nombre:15} {numerador:>3}/{den} = {num(numerador / den)}")
print(f"  {'suma':15} {sum(caminos.values()):>3}/{den} = 1")


---
# 4. Contar los casos: la grilla de resultados ordenados

*Apunte, secciones 5.1, 5.2 y 7.6.*

El enfoque clásico necesita contar los casos posibles. Con dos extracciones, cada resultado ordenado
es un par: qué salió primero y qué salió segundo. La grilla los muestra todos. Las filas son la
primera bolilla y las columnas la segunda. Sin reposición, la diagonal no existe: una bolilla no
puede salir dos veces. Con 8 bolillas son `8 × 7 = 56` casillas, y cada **pareja** sin orden aparece
dos veces, una a cada lado de la diagonal: `56 / 2 = 28`.

Las casillas coral son "blanca y blanca". Contalas: con 5 blancas son `5 × 4 = 20` de 56, el
`20/56 = 0,3571` que el apunte repite. Activá la reposición y la diagonal vuelve: `25/64 = 0,3906`,
que es el error 4 de la sección 10.


In [ ]:
#@title Todos los resultados ordenados de dos extracciones { run: "auto", display-mode: "form" }

blancas = 5  #@param {type:"slider", min:1, max:8, step:1}
negras = 3  #@param {type:"slider", min:1, max:8, step:1}
reposicion = False  #@param {type:"boolean"}

total = blancas + negras
bolillas = ["B"] * blancas + ["N"] * negras
ordenados = total * total if reposicion else total * (total - 1)
favorables = blancas * blancas if reposicion else blancas * (blancas - 1)

fig, ax = plt.subplots(figsize=(6.8, 6.8))
for i, primera in enumerate(bolillas):
    for j, segunda in enumerate(bolillas):
        y = total - 1 - i
        if i == j and not reposicion:
            ax.add_patch(Rectangle((j, y), 0.94, 0.94, facecolor="none", edgecolor="#B5B5B5",
                                   hatch="////", linewidth=0.8))
            continue
        ambas = primera == "B" and segunda == "B"
        ax.add_patch(Rectangle((j, y), 0.94, 0.94, facecolor=CORAL if ambas else CREMA,
                               edgecolor=TINTA, linewidth=1))
        ax.text(j + 0.47, y + 0.47, f"{primera}{segunda}", ha="center", va="center", fontsize=8,
                color="white" if ambas else TINTA, weight="bold" if ambas else "normal")
ax.set_xlim(-0.1, total + 0.1); ax.set_ylim(-0.1, total + 0.1)
ax.set_xticks(np.arange(total) + 0.47); ax.set_yticks(np.arange(total) + 0.47)
ax.set_xticklabels([f"{b}{k + 1}" for k, b in enumerate(bolillas)], fontsize=8)
ax.set_yticklabels([f"{b}{k + 1}" for k, b in enumerate(bolillas)][::-1], fontsize=8)
ax.set_xlabel("segunda bolilla"); ax.set_ylabel("primera bolilla"); ax.grid(False)
ax.xaxis.set_label_position("top"); ax.xaxis.tick_top()
for lado in ax.spines.values():
    lado.set_visible(False)
ax.set_title(f"{ordenados} resultados ordenados, {favorables} con dos blancas", pad=28)
plt.show()

if reposicion:
    print(f"Con reposición: {total} × {total} = {ordenados} resultados ordenados; la diagonal cuenta.")
else:
    print(f"Sin reposición: {total} × {total - 1} = {ordenados} resultados ordenados; la diagonal no existe.")
    print(f"Parejas sin orden: cada una aparece dos veces, así que son {ordenados} / 2 = {ordenados // 2}.")
print(f"P(blanca y blanca) = {frac(favorables, ordenados)}")
otro = blancas * (blancas - 1) / (total * (total - 1)) if reposicion else blancas * blancas / (total * total)
print(f"Con la otra regla daría {num(otro)}. La diferencia es el error 4 de la sección 10: "
      "multiplicar marginales cuando la segunda depende de la primera.")


---
# 5. Tres herramientas para ver el espacio muestral

*Apunte, secciones 4.1, 4.3 y 6.1 a 6.4.*

La tabla de contingencia, el árbol y el diagrama de Venn muestran el mismo mazo. El evento A es "la
carta es del palo elegido" y el evento B es la característica de la derecha. Buscá el mismo número
en los tres paneles: la celda coral de la tabla, la punta del camino que pasa por A y por B, y la
zona común del Venn.

Cambiá el palo: el 3 se mueve pero la tabla siempre cierra en 40. Cambiá B a `as`: la celda pasa a
1 y los círculos casi no se tocan. Y elegí como B **otro palo**: la celda da 0, los círculos se
separan y la unión se calcula sin restar nada. Eso es un par de eventos **mutuamente excluyentes**,
y el apunte lo remarca: excluyentes quiere decir dependientes, porque saber que salió A hace
imposible B.


In [ ]:
#@title Tabla, árbol y Venn del mismo mazo { run: "auto", display-mode: "form" }

palo = "espadas"  #@param ["oros", "copas", "espadas", "bastos"]
caracteristica = "figura"  #@param ["figura", "número", "as", "rey", "oros", "copas", "espadas", "bastos"]

PALOS = ["oros", "copas", "espadas", "bastos"]
VALORES = [1, 2, 3, 4, 5, 6, 7, 10, 11, 12]
mazo = [(p, v) for p in PALOS for v in VALORES]
if caracteristica in PALOS:
    cumple_B = lambda p, v: p == caracteristica
else:
    cumple_B = {"figura": lambda p, v: v >= 10, "número": lambda p, v: v <= 7,
                "as": lambda p, v: v == 1, "rey": lambda p, v: v == 12}[caracteristica]

N = len(mazo)
nA = sum(1 for p, v in mazo if p == palo)
nB = sum(1 for p, v in mazo if cumple_B(p, v))
nAB = sum(1 for p, v in mazo if p == palo and cumple_B(p, v))
A, B = palo, caracteristica

fig, (ax_t, ax_a, ax_v) = plt.subplots(1, 3, figsize=(15, 4.6))
for ax in (ax_t, ax_a, ax_v):
    ax.axis("off")

# --- tabla de contingencia
filas = [[B, nAB, nB - nAB, nB],
         [f"no {B}", nA - nAB, N - nA - nB + nAB, N - nB],
         ["Total", nA, N - nA, N]]
for j, texto in enumerate(["", A, f"no {A}", "Total"]):
    ax_t.add_patch(Rectangle((j, 3), 1, 1, facecolor=TEAL if j else CREMA, edgecolor=TINTA))
    ax_t.text(j + 0.5, 3.5, texto, ha="center", va="center", color="white", weight="bold")
for i, fila in enumerate(filas):
    y = 2 - i
    for j, valor in enumerate(fila):
        destacada = (i == 0 and j == 1)
        margen = j == 3 or i == 2
        color = TEAL if j == 0 else CORAL if destacada else TEAL_PALIDO if margen else CREMA
        ax_t.add_patch(Rectangle((j, y), 1, 1, facecolor=color, edgecolor=TINTA))
        ax_t.text(j + 0.5, y + 0.5, str(valor), ha="center", va="center",
                  color="white" if j == 0 or destacada else TINTA,
                  weight="bold" if j == 0 or margen or destacada else "normal")
ax_t.set_xlim(0, 4); ax_t.set_ylim(0, 4.6); ax_t.set_title("Tabla de contingencia")

# --- árbol
ax_a.set_xlim(0, 10); ax_a.set_ylim(0, 10)
ax_a.plot(0.5, 5, "o", color=TINTA, markersize=8)
ramas = [(A, nA, [(B, nAB), (f"no {B}", nA - nAB)]),
         (f"no {A}", N - nA, [(B, nB - nAB), (f"no {B}", N - nA - nB + nAB)])]
for (nombre, cuenta, hijos), y in zip(ramas, [7.5, 2.5]):
    ax_a.plot([0.7, 3.3], [5, y], color=TINTA, linewidth=1.5)
    ax_a.plot(3.5, y, "o", color=TEAL, markersize=8)
    ax_a.text(1.9, (5 + y) / 2 + 0.4, f"{cuenta}/{N}", ha="center", fontsize=9)
    ax_a.text(3.5, y + 0.7, nombre, ha="center", fontsize=9)
    for (hn, hc), dy in zip(hijos, [1.4, -1.4]):
        y2 = y + dy
        ax_a.plot([3.7, 6.8], [y, y2], color=TINTA, linewidth=1.5)
        ax_a.plot(7.0, y2, "o", color=CORAL if hn == B else MOSTAZA, markersize=8)
        ax_a.text(5.3, (y + y2) / 2 + 0.35, f"{hc}/{cuenta}", ha="center", fontsize=9)
        ax_a.text(7.4, y2, f"{hn}: {hc}/{N}", va="center", fontsize=9)
ax_a.set_title("Árbol")

# --- diagrama de Venn
ax_v.set_xlim(0, 10); ax_v.set_ylim(0, 7)
ax_v.add_patch(Rectangle((0.3, 0.3), 9.4, 6.2, facecolor="none", edgecolor=TINTA, linewidth=1.3))
separacion = 2.4 if nAB else 4.6
ax_v.add_patch(Circle((5 - separacion / 2, 3.4), 2.2, facecolor=TEAL, alpha=0.6, edgecolor=TINTA))
ax_v.add_patch(Circle((5 + separacion / 2, 3.4), 2.2, facecolor=MOSTAZA, alpha=0.6, edgecolor=TINTA))
if nAB:
    posiciones = [(5 - separacion / 2 - 1.1, nA - nAB), (5.0, nAB), (5 + separacion / 2 + 1.1, nB - nAB)]
else:
    posiciones = [(5 - separacion / 2, nA), (5 + separacion / 2, nB)]
for xx, valor in posiciones:
    ax_v.text(xx, 3.4, str(valor), ha="center", va="center", fontsize=15, weight="bold")
ax_v.text(0.7, 0.6, f"ninguno de los dos: {N - nA - nB + nAB}", fontsize=10)
ax_v.text(5 - separacion / 2, 6.0, f"A: {A}", ha="center", fontsize=10)
ax_v.text(5 + separacion / 2, 6.0, f"B: {B}", ha="center", fontsize=10)
ax_v.set_title("Diagrama de Venn")
plt.tight_layout(); plt.show()

print(f"P(A) = {frac(nA, N)}      P(B) = {frac(nB, N)}")
print(f"P(A y B) = {frac(nAB, N)}   ← la celda coral, la punta del camino A→B y la zona común")
print(f"P(A o B) = P(A) + P(B) − P(A y B) = {nA}/{N} + {nB}/{N} − {nAB}/{N} = {frac(nA + nB - nAB, N)}")
print(f"P(B | A) = {frac(nAB, nA)}   ← la rama del árbol que sale de A")
if nAB == 0:
    print("\nA y B no comparten ninguna carta: son mutuamente excluyentes. Y por eso son dependientes:")
    print(f"P(B) = {num(nB / N)} pero P(B | A) = 0. Saber que salió A cambia la probabilidad de B.")
elif nAB == nA == nB:
    print("\nA y B son el mismo evento.")
else:
    print(f"\nA y B son compatibles: {nAB} carta(s) cumplen las dos condiciones.")


---
# 6. El denominador es una decisión

*Apunte, sección 6.5.*

En el mazo el total es 40 y no hay nada que discutir. En datos reales el total depende de qué se
hace con los **faltantes**. En el relevamiento, 10 de los 37 hogares no tienen el dato de cobertura
de salud. Si se cuentan en el total, la probabilidad de tener cobertura es una. Si se descartan, es
otra. Las dos preguntas son válidas; lo que está mal es no decir cuál se contestó.

El gráfico muestra las dos respuestas para cada localidad y para el total. La opción elegida se ve
saturada; la otra queda en gris. Fijate cuánto cambia el resultado con solo mover el denominador.


In [ ]:
#@title Con cobertura de salud: dos denominadores { run: "auto", display-mode: "form" }

sin_dato = "incluir en el total"  #@param ["incluir en el total", "descartar"]

tabla = pd.crosstab(hogares["cobertura_salud"].fillna("sin dato"), hogares["localidad"],
                    margins=True, margins_name="Total")
tabla = tabla.reindex(["si", "no", "sin dato", "Total"])
tabla.index = ["Con cobertura", "Sin cobertura", "Sin dato", "Total"]
tabla.index.name = None; tabla.columns.name = None
display(tabla)

grupos = ["Concepción", "Córdoba", "Ramallo", "Total"]
incluyendo, descartando, denominadores = [], [], {}
for g in grupos:
    parte = hogares if g == "Total" else hogares[hogares["localidad"] == g]
    con = (parte["cobertura_salud"] == "si").sum()
    contestaron = parte["cobertura_salud"].notna().sum()
    incluyendo.append(con / len(parte))
    descartando.append(con / contestaron)
    denominadores[g] = (con, len(parte), contestaron)

elegida = incluyendo if sin_dato == "incluir en el total" else descartando
fig, ax = plt.subplots(figsize=(9.5, 4.4))
x = np.arange(len(grupos))
ax.bar(x - 0.19, incluyendo, 0.38, color=TEAL if elegida is incluyendo else "#C9C9C9",
       label="incluyendo los sin dato en el total")
ax.bar(x + 0.19, descartando, 0.38, color=CORAL if elegida is descartando else "#C9C9C9",
       label="descartando los sin dato")
for xx, v in zip(list(x - 0.19) + list(x + 0.19), incluyendo + descartando):
    ax.text(xx, v + 0.015, num(v, 3), ha="center", fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(grupos); ax.set_ylim(0, 0.9)
ax.set_ylabel("P(con cobertura)"); ax.legend(loc="upper left")
ax.set_title("La misma pregunta, dos totales")
plt.show()

con, n, contestaron = denominadores["Total"]
if sin_dato == "incluir en el total":
    print(f"Sobre los {n} hogares relevados: P(con cobertura) = {frac(con, n)}")
    print("Se está afirmando algo sobre todos los hogares visitados, incluidos los que no contestaron.")
else:
    print(f"Sobre los {contestaron} hogares que contestaron: P(con cobertura) = {frac(con, contestaron)}")
    print("Se está afirmando algo solo sobre los que contestaron. Hay que decirlo.")
print(f"La diferencia entre las dos respuestas es de {num(100 * abs(con / n - con / contestaron), 1)} puntos.")


---
# 7. Marginal, condicional, compuesta y total

*Apunte, secciones 7.1 a 7.9.*

Con dos eventos sobre la misma tabla salen las cuatro probabilidades de la unidad. La **marginal**
es la de un evento solo, sobre el total. La **condicional** achica el total: se queda con las filas
que cumplen la condición y cuenta ahí. La **compuesta** es la celda donde los dos se cruzan. Y la
**total**, que el programa también llama unión, suma las dos marginales y resta la compuesta para no
contar dos veces la intersección.

A la izquierda, la tabla de A contra B. La franja de B está resaltada porque condicionar por B es
quedarse solo con esa franja. A la derecha, el error más frecuente de la unidad: `P(A|B)` y `P(B|A)`
tienen el mismo numerador y distinto denominador, así que casi nunca son iguales.

El denominador vuelve a ser una decisión. Con `descartar`, los hogares sin dato en la variable de A
o de B salen del total, y se reproducen los números de la sección 7.4 del apunte, como el `3/7` de
Córdoba. Con `incluir`, el total es 37 y un hogar sin dato cuenta como que no cumple la condición.
El 54 % de hogares sin gas de la sección 3.6 es el complemento de `servicios incluye gas`.


In [ ]:
#@title Dos eventos sobre el relevamiento { run: "auto", display-mode: "form" }

A = "cobertura_salud = si"  #@param ["localidad = Ramallo", "localidad = Córdoba", "localidad = Concepción", "cobertura_salud = si", "cobertura_salud = no", "satisfaccion = buena", "satisfaccion = mala", "personas ≥ 4", "personas ≤ 2", "servicios incluye gas", "servicios incluye cloacas"]
B = "localidad = Córdoba"  #@param ["localidad = Ramallo", "localidad = Córdoba", "localidad = Concepción", "cobertura_salud = si", "cobertura_salud = no", "satisfaccion = buena", "satisfaccion = mala", "personas ≥ 4", "personas ≤ 2", "servicios incluye gas", "servicios incluye cloacas"]
sin_dato = "descartar"  #@param ["descartar", "incluir en el total"]

columnas = [A.split(" ")[0], B.split(" ")[0]]
base = hogares.dropna(subset=columnas) if sin_dato == "descartar" else hogares


def evento(descripcion):
    columna, operador, valor = descripcion.split(" ", 2)
    s = base[columna]
    if operador == "=":
        return (s == valor).to_numpy()
    if operador == "≥":
        return (s >= int(valor)).to_numpy()
    if operador == "≤":
        return (s <= int(valor)).to_numpy()
    return s.fillna("").str.contains(valor).to_numpy()


def en_dos_lineas(texto):
    return texto.replace(" = ", "\n= ").replace(" incluye ", "\nincluye ")


a, b = evento(A), evento(B)
N = len(base)
nA, nB, nAB = int(a.sum()), int(b.sum()), int((a & b).sum())

fig, (ax_t, ax_b) = plt.subplots(1, 2, figsize=(13, 4.6), gridspec_kw={"width_ratios": [1.3, 1]})
ax_t.axis("off")
celdas = [[en_dos_lineas(B), nAB, nB - nAB, nB],
          ["no " + en_dos_lineas(B), nA - nAB, N - nA - nB + nAB, N - nB],
          ["Total", nA, N - nA, N]]
for j, texto in enumerate(["", en_dos_lineas(A), "no " + en_dos_lineas(A), "Total"]):
    ax_t.add_patch(Rectangle((j, 3), 1, 1, facecolor=TEAL if j else CREMA, edgecolor=TINTA))
    ax_t.text(j + 0.5, 3.5, texto, ha="center", va="center", color="white", weight="bold", fontsize=8)
for i, fila in enumerate(celdas):
    y = 2 - i
    for j, valor in enumerate(fila):
        en_franja = i == 0
        if j == 0:
            color, texto_color = TEAL, "white"
        elif en_franja:
            color, texto_color = (CORAL if j == 1 else TEAL_PALIDO), ("white" if j == 1 else TINTA)
        else:
            color, texto_color = CREMA, "#9AA3B1"
        ax_t.add_patch(Rectangle((j, y), 1, 1, facecolor=color, edgecolor=TINTA))
        ax_t.text(j + 0.5, y + 0.5, str(valor), ha="center", va="center", color=texto_color,
                  fontsize=8 if j == 0 else 12, weight="bold" if j == 0 or en_franja else "normal")
ax_t.add_patch(Rectangle((1, 2), 3, 1, facecolor="none", edgecolor=CORAL, linewidth=3))
ax_t.set_xlim(0, 4); ax_t.set_ylim(0, 4.6)
ax_t.set_title(f"Condicionar por B recorta el total: de {N} a {nB}")

p_A_dado_B = nAB / nB if nB else np.nan
p_B_dado_A = nAB / nA if nA else np.nan
ax_b.bar(["P(A | B)", "P(B | A)"], [np.nan_to_num(p_A_dado_B), np.nan_to_num(p_B_dado_A)],
         color=[CORAL, MOSTAZA], width=0.55)
for i, (v, k, d) in enumerate([(p_A_dado_B, nAB, nB), (p_B_dado_A, nAB, nA)]):
    ax_b.text(i, (0 if np.isnan(v) else v) + 0.03, f"{k}/{d}" if d else "—", ha="center", fontsize=13)
ax_b.set_ylim(0, 1.12); ax_b.set_title(f"Mismo numerador ({nAB}), distinto denominador")
plt.tight_layout(); plt.show()

print(f"A: {A}      B: {B}      total: {N} hogares"
      + (f" (se descartaron {len(hogares) - N} sin dato)" if N < len(hogares) else "") + "\n")
print(f"Marginal      P(A)     = {frac(nA, N)}")
print(f"Marginal      P(B)     = {frac(nB, N)}")
print(f"Compuesta     P(A y B) = {frac(nAB, N)}")
print(f"Condicional   P(A | B) = {frac(nAB, nB) if nB else '— (B no ocurre nunca)'}")
print(f"Condicional   P(B | A) = {frac(nAB, nA) if nA else '— (A no ocurre nunca)'}")
print(f"Total (unión) P(A o B) = P(A) + P(B) − P(A y B) = {frac(nA + nB - nAB, N)}")
if nA and nB:
    print(f"\nControl: P(A y B) = P(A) × P(B | A) = {num(nA / N)} × {num(p_B_dado_A)} = {num(nA / N * p_B_dado_A)}")
    if A == B:
        print("A y B son el mismo evento: P(A | B) = 1 y no hay nada que comparar.")
    else:
        diferencia = p_A_dado_B - nA / N
        if abs(diferencia) < 1e-9:
            print(f"P(A | B) = P(A) = {num(nA / N)}: en esta tabla, saber B no cambia nada.")
        else:
            print(f"P(A | B) = {num(p_A_dado_B)} contra P(A) = {num(nA / N)}: saber B cambia la probabilidad de A "
                  f"en {num(abs(diferencia), 3)}. Con {nAB} casos en la celda, esa diferencia también podría ser "
                  "casualidad del muestreo. Decidirlo es un test de independencia, y llega al final de la asignatura.")


---
# 8. La ley de probabilidad total: sumar caminos

*Apunte, sección 8.2.*

El nombre "probabilidad total" tiene dos usos. En el programa de la asignatura es la probabilidad
de la unión, la del bloque 7. En la bibliografía, la **ley de probabilidad total** es otra cosa: la
probabilidad de un evento sumando los caminos que llevan a él, cuando esos caminos parten el espacio
en pedazos que no se pisan.

El árbol del bolillero lo muestra. La probabilidad de que la **segunda** bolilla sea blanca es la
suma de dos caminos: primera blanca y después blanca, o primera negra y después blanca. Cada camino
es una compuesta, y la suma vale porque los dos caminos son excluyentes. Sin reposición, el
resultado sorprende: `20/56 + 15/56 = 35/56 = 5/8`, la misma probabilidad que tenía la primera.


In [ ]:
#@title El árbol del bolillero y la suma de caminos { run: "auto", display-mode: "form" }

blancas = 5  #@param {type:"slider", min:1, max:12, step:1}
negras = 3  #@param {type:"slider", min:1, max:12, step:1}
reposicion = False  #@param {type:"boolean"}

total = blancas + negras
composicion = {"blanca": blancas, "negra": negras}
den_2 = total if reposicion else total - 1
den = total * den_2


def quedan(color_1, color_2):
    """Cuántas bolillas de color_2 quedan para la segunda extracción, si la primera fue color_1."""
    if reposicion:
        return composicion[color_2]
    return composicion[color_2] - (color_1 == color_2)


caminos = {(c1, c2): composicion[c1] * quedan(c1, c2) for c1 in composicion for c2 in composicion}
num_blanca = sum(v for (c1, c2), v in caminos.items() if c2 == "blanca")

fig, ax = plt.subplots(figsize=(11, 4.8))
ax.axis("off"); ax.set_xlim(0, 13.5); ax.set_ylim(0, 10)
ax.plot(0.6, 5, "o", color=TINTA, markersize=9); ax.text(0.6, 5.6, "inicio", ha="center", fontsize=9)
colores = {"blanca": "white", "negra": TINTA}
for c1, y in zip(composicion, [7.5, 2.5]):
    ax.plot([0.8, 3.6], [5, y], color=TINTA, linewidth=1.6)
    ax.plot(3.8, y, "o", markersize=13, markerfacecolor=colores[c1], markeredgecolor=TINTA)
    ax.text(2.0, (5 + y) / 2 + 0.4, f"{composicion[c1]}/{total}", ha="center", fontsize=10)
    for c2, dy in zip(composicion, [1.5, -1.5]):
        y2 = y + dy
        destacado = c2 == "blanca"
        alfa = 1 if destacado else 0.45
        ax.plot([4.0, 7.4], [y, y2], color=CORAL if destacado else TINTA,
                linewidth=2.6 if destacado else 1.2, alpha=alfa)
        ax.plot(7.6, y2, "o", markersize=13, markerfacecolor=colores[c2], markeredgecolor=TINTA, alpha=alfa)
        ax.text(5.7, (y + y2) / 2 + 0.4, f"{quedan(c1, c2)}/{den_2}", ha="center", fontsize=10, alpha=alfa)
        ax.text(8.2, y2, f"{c1}, {c2}:  {composicion[c1]}/{total} × {quedan(c1, c2)}/{den_2} = "
                f"{caminos[(c1, c2)]}/{den}", va="center", fontsize=10,
                color=CORAL if destacado else TINTA, alpha=alfa, weight="bold" if destacado else "normal")
ax.set_title(f"P(2ª blanca) = suma de los dos caminos resaltados = {num_blanca}/{den} = {num(num_blanca / den)}")
plt.show()

sumandos = " + ".join(f"{caminos[(c1, 'blanca')]}/{den}" for c1 in composicion)
print(f"P(2ª blanca) = {sumandos} = {num_blanca}/{den} = {frac(blancas, total)}")
print(f"P(1ª blanca) = {frac(blancas, total)}")
print("Las dos coinciden, con o sin reposición: sin saber qué salió primero, la segunda es como la primera.")
print(f"Suma de los cuatro caminos: {sum(caminos.values())}/{den} = 1")


---
## Cierre

Ocho formularios, y en todos se puede volver a cambiar algo:

1. El mazo como espacio muestral: el evento, su complemento, y la suma que da 1.
2. La proporción que se aplaca, el desbalance absoluto que no, y la racha que no cambia nada.
3. La reposición como la diferencia entre independiente y dependiente: `4/7` contra `5/7`.
4. La grilla de los 56 resultados ordenados, y el `20/56` que recorre la unidad.
5. Tabla, árbol y Venn: el mismo número en los tres, y dos eventos que no se tocan.
6. El denominador como decisión, con los diez hogares sin dato.
7. Las cuatro probabilidades sobre datos reales, y `P(A|B)` contra `P(B|A)`.
8. La ley de probabilidad total: la suma de caminos del árbol.

Quedan afuera, porque se entienden leyendo, los dos tipos de incertidumbre de la sección 2.1, el
enfoque subjetivo de la sección 3.3, el ejemplo del vuelo de la sección 3.4 y la demostración de las
dos fórmulas de la sección 9.

> **Apunte de la unidad 1:**
> [E8102_Apunte_U1_Probabilidad.pdf](https://github.com/lautarodibartolo-ae/e8102-exploracion-de-datos/blob/main/bloque-1-probabilidad-y-variables-aleatorias/E8102_Apunte_U1_Probabilidad.pdf)
